# Part 2: Gateway and MCP Tools Integration
## Transform Backend APIs into Secure Agent Tools

## Overview

In this notebook, you'll transform your backend APIs into secure, standardized tools that your SRE agents can use. We'll create an Amazon Bedrock AgentCore Gateway that implements the Model Context Protocol (MCP), providing a secure bridge between your agents and infrastructure APIs.

By the end of this notebook, your agents will be able to securely invoke real tools to investigate infrastructure issues.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Gateway Integration                                                              |
| Agent type          | Multi-Agent with MCP Tools                                                      |
| Agentic Framework   | MCP Protocol with AgentCore Gateway                                             |
| LLM models          | Amazon Nova Pro, Anthropic Claude Sonnet 3.7                                   |
| Tutorial components | AgentCore Gateway, Cognito Auth, MCP Tools, API Security                       |
| Tutorial vertical   | DevOps/SRE                                                                       |
| Example complexity  | Intermediate                                                                     |
| Duration           | 45-60 minutes                                                                    |

### Learning Objectives

- **Understand MCP Protocol**: Learn how agents communicate with tools securely
- **Create AgentCore Gateway**: Deploy secure API gateway for agent tools
- **Set Up Authentication**: Configure Amazon Cognito for OAuth 2.0 security
- **Transform APIs to Tools**: Convert backend services into MCP-compliant tools
- **Test Tool Integration**: Validate agents can invoke tools through gateway
- **Understand Security Model**: Learn inbound/outbound authentication patterns

### Architecture Overview

We're building the secure gateway layer that enables agents to access your infrastructure APIs:

```
┌─────────────────────────────────────────────────────────────────┐
│                    SRE Agent System                             │
│  ┌──────────────┐    ┌─────────────────────────────────────┐   │
│  │  Supervisor  │    │         Specialist Agents           │   │
│  │    Agent     │    │  ┌─────┬─────┬─────┬──────────────┐ │   │
│  └──────┬───────┘    │  │ K8s │Logs │Metr-│   Runbooks   │ │   │
│         │            │  │Agent│Agent│ics  │    Agent     │ │   │
│         │            │  └─────┴─────┴─────┴──────────────┘ │   │
│         │            └─────────────┬───────────────────────┘   │
└─────────┼──────────────────────────┼───────────────────────────┘
          │                          │
          │         🔒 HTTPS + OAuth │
          └──────────────┬───────────┘
                         │
          ┌──────────────▼──────────────────────────────────────┐
          │           AgentCore Gateway                         │
          │  ┌─────────────────────────────────────────────┐   │
          │  │            MCP Protocol Layer               │   │
          │  │  • ListTools API                            │   │
          │  │  • InvokeTools API                          │   │
          │  │  • Schema Validation                        │   │
          │  └─────────────┬───────────────────────────────┘   │
          │  ┌─────────────▼───────────────────────────────┐   │
          │  │         Security & Auth Layer              │   │
          │  │  • Amazon Cognito (OAuth 2.0)              │   │
          │  │  • JWT Token Validation                     │   │
          │  │  • IAM Role Authentication                  │   │
          │  └─────────────┬───────────────────────────────┘   │
          └────────────────┼─────────────────────────────────────┘
                           │
                  ┌────────▼────────┐
                  │  Backend APIs   │
                  │                 │
                  │ • K8s API       │
                  │ • Logs API      │
                  │ • Metrics API   │
                  │ • Runbooks API  │
                  └─────────────────┘
```

## Prerequisites Check

Let's ensure Notebook 1 was completed successfully and all backend services are running.

In [ ]:
# Import workshop utilities and check prerequisites
import sys
import os
from pathlib import Path

# Add workshop helpers to path
sys.path.append(str(Path().absolute().parent / "helpers"))
sys.path.append(str(Path().absolute().parent.parent.parent))

from workshop_utils import load_workshop_config, get_aws_account_id, get_private_ip
from validation_helpers import WorkshopValidator, print_validation_summary
from sre_scenarios import SREScenarios

# Load environment variables
from dotenv import load_dotenv
env_file = Path().absolute().parent.parent.parent / "sre_agent" / ".env"
load_dotenv(env_file)

print("✓ Workshop utilities imported")
print(f"✓ Environment loaded from {env_file}")

In [ ]:
# Validate prerequisites from Notebook 1
config = load_workshop_config()
validator = WorkshopValidator(config)

print("Validating Notebook 1 prerequisites...")
print("═" * 50)

# Check environment setup
env_results = validator.validate_environment_setup()

# Check backend services are running
private_ip = get_private_ip()
use_ssl = os.path.exists("/opt/ssl/fullchain.pem") and os.path.exists("/opt/ssl/privkey.pem")
protocol = "https" if use_ssl else "http"

backend_urls = [
    f"{protocol}://{private_ip}:8011",  # Kubernetes
    f"{protocol}://{private_ip}:8012",  # Logs
    f"{protocol}://{private_ip}:8013",  # Metrics
    f"{protocol}://{private_ip}:8014"   # Runbooks
]

backend_results = validator.validate_backend_services(backend_urls)

prerequisite_results = {
    "Environment": env_results,
    "Backend Services": backend_results
}

print_validation_summary(prerequisite_results)

# Store for later use
healthy_services = sum(backend_results.values())
if healthy_services < 3:
    print("\n⚠️  Warning: Less than 3 backend services are healthy.")
    print("   Please return to Notebook 1 and restart the backend services.")
else:
    print(f"\n✅ Prerequisites met! {healthy_services}/4 backend services healthy.")
    print("   Ready to proceed with Gateway integration.")

## Understanding the Model Context Protocol (MCP)

Before we build the gateway, let's understand what MCP provides and why it's important for AI agents.

In [ ]:
print("🔧 Model Context Protocol (MCP) Overview")
print("═" * 60)

print("\n🎯 What is MCP?")
print("   MCP is a standardized protocol for AI agents to interact with tools.")
print("   It provides a uniform interface regardless of the underlying API.")

print("\n🔧 Core MCP Operations:")
mcp_operations = {
    "ListTools": "Discover available tools and their capabilities",
    "InvokeTools": "Execute specific tools with parameters", 
    "GetSchema": "Retrieve tool input/output specifications"
}

for operation, description in mcp_operations.items():
    print(f"   • {operation:<12} {description}")

print("\n🏗️ AgentCore Gateway Benefits:")
benefits = [
    "Transforms any REST API into MCP tools",
    "Provides OAuth 2.0 security and authentication", 
    "Handles schema validation and error handling",
    "Enables tool discovery and introspection",
    "Supports multiple authentication methods"
]

for benefit in benefits:
    print(f"   ✓ {benefit}")

print("\n🔄 Agent-Gateway Communication Flow:")
print("   1. Agent requests available tools (ListTools)")
print("   2. Gateway returns tool schemas and capabilities")
print("   3. Agent selects appropriate tool for task")
print("   4. Agent invokes tool with parameters (InvokeTools)")
print("   5. Gateway validates, authenticates, and forwards request")
print("   6. Backend API processes request and returns data")
print("   7. Gateway formats response and returns to agent")

print("\n🛡️ Security Model:")
print("   • Inbound Auth: OAuth tokens validate agent access")
print("   • Outbound Auth: IAM roles access backend services")
print("   • HTTPS Required: All communication encrypted")
print("   • Schema Validation: Prevent malicious inputs")

## Step 1: Create Amazon Cognito Authentication

We'll start by setting up OAuth 2.0 authentication using Amazon Cognito. This provides secure access control for our gateway.

In [ ]:
# Create Cognito User Pool for gateway authentication
import boto3
import time
from workshop_utils import create_cognito_user_pool, get_cognito_token

print("🔐 Creating Amazon Cognito Authentication")
print("═" * 50)

region = config.get('aws', {}).get('region', 'us-east-1')
pool_name = f"{config.get('names', {}).get('prefix', 'sre-workshop')}-pool"

print(f"Region: {region}")
print(f"Pool Name: {pool_name}")

try:
    # Create Cognito resources
    print("\n📋 Creating Cognito User Pool...")
    user_pool_id, client_id, client_secret = create_cognito_user_pool(
        pool_name=pool_name,
        region=region
    )
    
    print(f"✅ User Pool Created: {user_pool_id}")
    print(f"✅ App Client Created: {client_id}")
    print(f"✅ Client Secret Generated: {client_secret[:8]}...")
    
    # Create discovery URL for OIDC
    discovery_url = f'https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
    print(f"\n🔗 OIDC Discovery URL: {discovery_url}")
    
    # Store credentials for later use
    cognito_config = {
        'user_pool_id': user_pool_id,
        'client_id': client_id,
        'client_secret': client_secret,
        'discovery_url': discovery_url,
        'region': region
    }
    
    print("\n✅ Cognito authentication setup complete!")
    
except Exception as e:
    print(f"❌ Failed to create Cognito resources: {e}")
    print("\nPlease check:")
    print("   • AWS credentials have Cognito permissions")
    print("   • Region is correct and service is available")
    print("   • No naming conflicts with existing resources")
    raise

In [ ]:
# Test OAuth token generation
print("🔑 Testing OAuth Token Generation")
print("═" * 40)

try:
    print("Requesting OAuth access token...")
    access_token = get_cognito_token(
        user_pool_id=cognito_config['user_pool_id'],
        client_id=cognito_config['client_id'],
        client_secret=cognito_config['client_secret'],
        region=region
    )
    
    print(f"✅ Access token generated: {access_token[:20]}...")
    print(f"✅ Token length: {len(access_token)} characters")
    
    # Store token for gateway testing
    cognito_config['access_token'] = access_token
    
    print("\n🎯 OAuth Flow Validated:")
    print("   ✓ Client credentials flow working")
    print("   ✓ Bearer token ready for gateway")
    print("   ✓ Ready for AgentCore Gateway integration")
    
except Exception as e:
    print(f"❌ OAuth token generation failed: {e}")
    print("\nThis might be temporary - Cognito domains take time to propagate.")
    print("We'll retry this when creating the gateway.")
    cognito_config['access_token'] = None

## Step 2: Create IAM Role for Gateway

The gateway needs IAM permissions to access your backend services and manage the gateway itself.

In [ ]:
# Create IAM role for AgentCore Gateway
from workshop_utils import create_iam_role

print("👤 Creating IAM Role for AgentCore Gateway")
print("═" * 50)

role_name = f"{config.get('names', {}).get('prefix', 'sre-workshop')}-gateway-role"

# Define service principals and policies
service_principals = [
    "bedrock-agentcore-gateway.amazonaws.com",  # AgentCore Gateway service
    "lambda.amazonaws.com"  # For potential Lambda integrations
]

required_policies = [
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    "arn:aws:iam::aws:policy/AmazonBedrockAgentCoreGatewayServiceRolePolicy"
]

try:
    print(f"Creating IAM role: {role_name}")
    print(f"Service principals: {', '.join(service_principals)}")
    
    gateway_role_arn = create_iam_role(
        role_name=role_name,
        service_principals=service_principals,
        policies=required_policies
    )
    
    print(f"✅ Gateway IAM role created: {gateway_role_arn}")
    
    # Validate role was created successfully
    iam_client = boto3.client('iam')
    role_details = iam_client.get_role(RoleName=role_name)
    
    print("\n🔍 Role Details:")
    print(f"   Role ARN: {role_details['Role']['Arn']}")
    print(f"   Created: {role_details['Role']['CreateDate']}")
    
    # List attached policies
    attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
    print(f"\n📋 Attached Policies ({len(attached_policies['AttachedPolicies'])}):")
    for policy in attached_policies['AttachedPolicies']:
        print(f"   ✓ {policy['PolicyName']}")
        
    print("\n✅ IAM role setup complete!")
    
except Exception as e:
    print(f"❌ Failed to create IAM role: {e}")
    print("\nPlease check:")
    print("   • AWS credentials have IAM permissions")
    print("   • Role name doesn't conflict with existing roles")
    print("   • Required policies exist in your account")
    raise

## Step 3: Create AgentCore Gateway

Now we'll create the AgentCore Gateway that will transform your backend APIs into MCP tools.

In [ ]:
# Create AgentCore Gateway
print("🌐 Creating Amazon Bedrock AgentCore Gateway")
print("═" * 55)

gateway_name = config.get('names', {}).get('gateway', 'sre-gateway')

# Configure gateway with Cognito authentication
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config['client_id']],
        "discoveryUrl": cognito_config['discovery_url']
    }
}

gateway_client = boto3.client('bedrock-agentcore-control', region_name=region)

try:
    print(f"Creating gateway: {gateway_name}")
    print(f"Authentication: Custom JWT (Cognito)")
    print(f"Protocol: MCP")
    
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration=auth_config,
        description='SRE Agent Workshop - Multi-Agent Infrastructure Tools'
    )
    
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    
    print(f"\n✅ Gateway Created Successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
    # Store gateway info
    gateway_config = {
        'gateway_id': gateway_id,
        'gateway_url': gateway_url,
        'gateway_name': gateway_name
    }
    
    print("\n⏳ Gateway is being deployed...")
    print("   This typically takes 2-3 minutes to become READY.")
    
except Exception as e:
    print(f"❌ Failed to create gateway: {e}")
    print("\nPlease check:")
    print("   • AWS credentials have AgentCore permissions")
    print("   • IAM role ARN is correct")
    print("   • Cognito configuration is valid")
    print("   • No naming conflicts with existing gateways")
    raise

In [ ]:
# Wait for gateway to become ready
print("⏳ Waiting for gateway to become ready...")
print("   This process typically takes 2-3 minutes.")

import time
from workshop_utils import wait_for_resource_ready

def check_gateway_ready():
    """Check if gateway is in READY state."""
    try:
        response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
        status = response['gateway']['status']
        print(f"   Gateway status: {status}")
        return status == 'READY'
    except Exception as e:
        print(f"   Error checking gateway: {e}")
        return False

# Wait up to 5 minutes for gateway to be ready
gateway_ready = wait_for_resource_ready(
    check_function=check_gateway_ready,
    max_wait_time=300,  # 5 minutes
    check_interval=15   # Check every 15 seconds
)

if gateway_ready:
    print("\n✅ Gateway is READY!")
    print(f"   You can now create gateway targets and test MCP tools.")
else:
    print("\n⚠️  Gateway is not ready yet, but we can continue.")
    print("   Gateway deployment continues in the background.")
    print("   We'll check status again when creating targets.")

# Get final gateway details
try:
    gateway_details = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    final_status = gateway_details['gateway']['status']
    
    print(f"\n🔍 Gateway Details:")
    print(f"   Status: {final_status}")
    print(f"   Created: {gateway_details['gateway'].get('createdAt', 'Unknown')}")
    print(f"   Protocol: {gateway_details['gateway']['protocolType']}")
    print(f"   Authorizer: {gateway_details['gateway']['authorizerType']}")
    
except Exception as e:
    print(f"   Could not get gateway details: {e}")

## Step 4: Create Gateway Targets (MCP Tools)

Now we'll transform each backend API into MCP tools by creating gateway targets. Each target maps to one of your backend services.

In [ ]:
# Define backend service targets and their MCP tool schemas
print("🔧 Defining Backend Service MCP Tool Schemas")
print("═" * 50)

# Define MCP tool schemas for each backend service
backend_targets = {
    "kubernetes_api": {
        "name": "KubernetesAPI",
        "description": "Kubernetes cluster operations and monitoring",
        "url": f"{protocol}://{private_ip}:8011",
        "tools": [
            {
                "name": "get_pod_status",
                "description": "Get status of pods in the cluster",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "namespace": {"type": "string", "description": "Kubernetes namespace"},
                        "pod_name": {"type": "string", "description": "Specific pod name (optional)"}
                    }
                }
            },
            {
                "name": "get_deployment_status",
                "description": "Get deployment status and replica information",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "namespace": {"type": "string", "description": "Kubernetes namespace"},
                        "deployment_name": {"type": "string", "description": "Deployment name (optional)"}
                    }
                }
            },
            {
                "name": "get_cluster_events",
                "description": "Get recent cluster events and warnings",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "limit": {"type": "integer", "description": "Number of events to retrieve", "default": 10},
                        "event_type": {"type": "string", "description": "Filter by event type (Warning, Normal)"}
                    }
                }
            }
        ]
    },
    
    "logs_api": {
        "name": "LogsAPI",
        "description": "Application log analysis and searching",
        "url": f"{protocol}://{private_ip}:8012",
        "tools": [
            {
                "name": "search_logs",
                "description": "Search application logs with query patterns",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query or pattern"},
                        "service": {"type": "string", "description": "Filter by service name"},
                        "limit": {"type": "integer", "description": "Number of log entries", "default": 20}
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "get_error_logs",
                "description": "Retrieve recent error and exception logs",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "service": {"type": "string", "description": "Filter by service name"},
                        "time_range": {"type": "string", "description": "Time range (1h, 24h, 7d)", "default": "1h"},
                        "limit": {"type": "integer", "description": "Number of errors", "default": 10}
                    }
                }
            }
        ]
    },
    
    "metrics_api": {
        "name": "MetricsAPI", 
        "description": "Performance metrics and resource monitoring",
        "url": f"{protocol}://{private_ip}:8013",
        "tools": [
            {
                "name": "get_performance_metrics",
                "description": "Get application performance metrics",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "service": {"type": "string", "description": "Service name"},
                        "metric_type": {"type": "string", "description": "Metric type (response_time, throughput, error_rate)"},
                        "time_range": {"type": "string", "description": "Time range", "default": "1h"}
                    }
                }
            },
            {
                "name": "get_resource_metrics",
                "description": "Get infrastructure resource usage metrics",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "resource_type": {"type": "string", "description": "Resource type (cpu, memory, disk, network)"},
                        "node": {"type": "string", "description": "Specific node name (optional)"}
                    }
                }
            }
        ]
    },
    
    "runbooks_api": {
        "name": "RunbooksAPI",
        "description": "Operational procedures and troubleshooting guides", 
        "url": f"{protocol}://{private_ip}:8014",
        "tools": [
            {
                "name": "search_runbooks",
                "description": "Search operational runbooks and procedures",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query"},
                        "category": {"type": "string", "description": "Runbook category (incident, troubleshooting, escalation)"}
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "get_escalation_procedures", 
                "description": "Get escalation procedures for incidents",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "severity": {"type": "string", "description": "Incident severity (low, medium, high, critical)"},
                        "service": {"type": "string", "description": "Affected service"}
                    },
                    "required": ["severity"]
                }
            }
        ]
    }
}

print(f"✅ Defined {len(backend_targets)} backend service targets:")
total_tools = 0
for target_id, target_config in backend_targets.items():
    tool_count = len(target_config['tools'])
    total_tools += tool_count
    print(f"   • {target_config['name']}: {tool_count} tools")
    
print(f"\n🔧 Total MCP tools to create: {total_tools}")

In [ ]:
# Create gateway targets for each backend service
print("🎯 Creating Gateway Targets (MCP Tools)")
print("═" * 45)

created_targets = {}
target_creation_results = {}

for target_id, target_config in backend_targets.items():
    print(f"\n📋 Creating target: {target_config['name']}")
    print(f"   URL: {target_config['url']}")
    print(f"   Tools: {len(target_config['tools'])}")
    
    # Configure target for HTTP API
    target_configuration = {
        "mcp": {
            "httpApi": {
                "baseUrl": target_config['url'],
                "toolSchema": {
                    "inlinePayload": target_config['tools']
                }
            }
        }
    }
    
    # Configure credentials (no authentication to backend for demo)
    credential_config = [{
        "credentialProviderType": "GATEWAY_IAM_ROLE"
    }]
    
    try:
        response = gateway_client.create_gateway_target(
            gatewayIdentifier=gateway_id,
            name=target_config['name'],
            description=target_config['description'],
            targetConfiguration=target_configuration,
            credentialProviderConfigurations=credential_config
        )
        
        target_name = response['name']
        target_status = response.get('status', 'UNKNOWN')
        
        created_targets[target_id] = {
            'name': target_name,
            'status': target_status,
            'tools': target_config['tools']
        }
        
        target_creation_results[target_id] = True
        
        print(f"   ✅ Target created: {target_name} (Status: {target_status})")
        
    except Exception as e:
        print(f"   ❌ Failed to create target: {e}")
        target_creation_results[target_id] = False
        
        # Continue with other targets
        continue

# Summary
successful_targets = sum(target_creation_results.values())
total_targets = len(backend_targets)

print(f"\n📊 Target Creation Summary:")
print(f"   Created: {successful_targets}/{total_targets} targets")
print(f"   Status: {'✅ All targets created' if successful_targets == total_targets else '⚠️  Some targets failed'}")

if successful_targets > 0:
    print(f"\n🎉 Gateway now has {successful_targets} MCP tool domains available!")
else:
    print(f"\n❌ No targets were created successfully. Please check the errors above.")

In [ ]:
# Wait for targets to become ready
if successful_targets > 0:
    print("⏳ Waiting for gateway targets to become ready...")
    print("   This typically takes 1-2 minutes per target.")
    
    def check_targets_ready():
        """Check if all targets are ready."""
        try:
            targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
            targets = targets_response.get('targets', [])
            
            ready_count = 0
            for target in targets:
                status = target.get('status', 'UNKNOWN')
                if status == 'READY':
                    ready_count += 1
            
            print(f"   Targets ready: {ready_count}/{len(targets)}")
            return ready_count == len(targets) and len(targets) > 0
            
        except Exception as e:
            print(f"   Error checking targets: {e}")
            return False
    
    # Wait up to 5 minutes for all targets to be ready
    targets_ready = wait_for_resource_ready(
        check_function=check_targets_ready,
        max_wait_time=300,  # 5 minutes
        check_interval=20   # Check every 20 seconds
    )
    
    if targets_ready:
        print("\n✅ All gateway targets are READY!")
        print("   MCP tools are now available for agent use.")
    else:
        print("\n⚠️  Not all targets are ready yet, but we can continue.")
        print("   Target deployment continues in the background.")
    
    # Get final target status
    try:
        final_targets = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
        print(f"\n🔍 Final Target Status:")
        for target in final_targets.get('targets', []):
            name = target.get('name', 'Unknown')
            status = target.get('status', 'Unknown')
            print(f"   {name}: {status}")
    except Exception as e:
        print(f"   Could not get final target status: {e}")
        
else:
    print("\n⚠️  No targets to wait for. Please review the target creation errors above.")

## Step 5: Test MCP Tools Access

Now let's test that our MCP tools are working by calling them directly through the gateway.

In [ ]:
# Test MCP ListTools endpoint
import requests
import json

print("🔍 Testing MCP Tools Access")
print("═" * 35)

# Get fresh OAuth token if needed
if not cognito_config.get('access_token'):
    print("🔑 Generating fresh OAuth token...")
    try:
        access_token = get_cognito_token(
            user_pool_id=cognito_config['user_pool_id'],
            client_id=cognito_config['client_id'],
            client_secret=cognito_config['client_secret'],
            region=region
        )
        cognito_config['access_token'] = access_token
        print(f"   ✅ Token generated: {access_token[:20]}...")
    except Exception as e:
        print(f"   ❌ Token generation failed: {e}")
        print("   Cannot test MCP tools without valid token.")
        access_token = None
else:
    access_token = cognito_config['access_token']
    print(f"✅ Using existing OAuth token: {access_token[:20]}...")

if access_token:
    # Test ListTools endpoint
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    list_tools_url = f"{gateway_url}/mcp/v1/tools/list"
    
    print(f"\n📋 Testing ListTools endpoint...")
    print(f"   URL: {list_tools_url}")
    
    try:
        response = requests.post(
            list_tools_url,
            headers=headers,
            json={},
            timeout=15,
            verify=False  # Skip SSL verification for demo
        )
        
        if response.status_code == 200:
            tools_data = response.json()
            tools = tools_data.get('tools', [])
            
            print(f"\n✅ ListTools successful! Found {len(tools)} tools:")
            
            # Organize tools by domain
            tool_domains = {}
            for tool in tools:
                tool_name = tool.get('name', 'unknown')
                # Extract domain from tool name (assumes format: Domain___tool_name)
                if '___' in tool_name:
                    domain = tool_name.split('___')[0]
                    if domain not in tool_domains:
                        tool_domains[domain] = []
                    tool_domains[domain].append(tool_name.split('___')[1])
                else:
                    if 'Other' not in tool_domains:
                        tool_domains['Other'] = []
                    tool_domains['Other'].append(tool_name)
            
            for domain, domain_tools in tool_domains.items():
                print(f"\n   🔧 {domain} ({len(domain_tools)} tools):")
                for tool_name in domain_tools:
                    print(f"      • {tool_name}")
            
            # Store tools for later testing
            available_tools = tools
            
        else:
            print(f"\n❌ ListTools failed: HTTP {response.status_code}")
            print(f"   Response: {response.text[:200]}...")
            available_tools = []
            
    except Exception as e:
        print(f"\n❌ ListTools request failed: {e}")
        available_tools = []
else:
    print("\n⚠️  Cannot test MCP tools without valid OAuth token.")
    available_tools = []

In [ ]:
# Test invoking a specific MCP tool
if available_tools:
    print("\n🎯 Testing Tool Invocation")
    print("═" * 30)
    
    # Find a simple tool to test (preferably a GET operation)
    test_tool = None
    for tool in available_tools:
        tool_name = tool.get('name', '')
        if 'get_pod_status' in tool_name or 'get_cluster_events' in tool_name:
            test_tool = tool
            break
    
    if not test_tool and available_tools:
        test_tool = available_tools[0]  # Use first available tool
    
    if test_tool:
        tool_name = test_tool['name']
        print(f"Testing tool: {tool_name}")
        
        # Prepare tool invocation
        invoke_url = f"{gateway_url}/mcp/v1/tools/call"
        
        # Use minimal arguments for testing
        tool_arguments = {}
        if 'get_cluster_events' in tool_name:
            tool_arguments = {"limit": 5}
        elif 'get_pod_status' in tool_name:
            tool_arguments = {"namespace": "default"}
        
        invoke_payload = {
            "name": tool_name,
            "arguments": tool_arguments
        }
        
        print(f"   Arguments: {tool_arguments}")
        print(f"   URL: {invoke_url}")
        
        try:
            response = requests.post(
                invoke_url,
                headers=headers,
                json=invoke_payload,
                timeout=20,
                verify=False
            )
            
            if response.status_code == 200:
                result_data = response.json()
                content = result_data.get('content', [{}])[0].get('text', 'No content')
                
                print(f"\n✅ Tool invocation successful!")
                print(f"   Tool: {tool_name}")
                print(f"   Response length: {len(content)} characters")
                print(f"   Response preview: {content[:200]}...")
                
                # Try to parse as JSON to show structured data
                try:
                    parsed_content = json.loads(content)
                    if isinstance(parsed_content, dict):
                        print(f"   Structured data keys: {list(parsed_content.keys())}")
                    elif isinstance(parsed_content, list):
                        print(f"   Array with {len(parsed_content)} items")
                except:
                    print(f"   Raw text response (not JSON)")
                
                print("\n🎉 MCP tool invocation working correctly!")
                
            else:
                print(f"\n❌ Tool invocation failed: HTTP {response.status_code}")
                print(f"   Response: {response.text[:300]}...")
                
        except Exception as e:
            print(f"\n❌ Tool invocation request failed: {e}")
    
    else:
        print("\n⚠️  No suitable test tool found in available tools.")
        
else:
    print("\n⚠️  Cannot test tool invocation - no tools available from ListTools.")

## Step 6: Validate Complete Gateway Setup

Let's run comprehensive validation to ensure everything is working correctly.

In [ ]:
# Run comprehensive validation of gateway setup
print("🔍 Comprehensive Gateway Validation")
print("═" * 40)

validation_results = {}

# 1. Cognito Setup Validation
print("\n1️⃣ Validating Cognito Setup...")
cognito_results = validator.validate_cognito_setup(
    user_pool_id=cognito_config['user_pool_id'],
    client_id=cognito_config['client_id']
)
validation_results["Cognito Authentication"] = cognito_results

# 2. Gateway Setup Validation  
print("\n2️⃣ Validating Gateway Setup...")
gateway_results = validator.validate_gateway_setup(gateway_id)
validation_results["AgentCore Gateway"] = gateway_results

# 3. MCP Tools Access Validation
print("\n3️⃣ Validating MCP Tools Access...")
if access_token:
    mcp_results = validator.validate_mcp_tools_access(gateway_url, access_token)
    validation_results["MCP Tools"] = mcp_results
else:
    validation_results["MCP Tools"] = {
        "list_tools_accessible": False,
        "tools_count": 0,
        "tool_invocation_works": False
    }

# 4. Backend Services Validation (from earlier)
validation_results["Backend Services"] = backend_results

# Print comprehensive summary
print_validation_summary(validation_results)

# Calculate overall success rate
all_checks = []
for section_results in validation_results.values():
    all_checks.extend(section_results.values())

success_rate = (sum(all_checks) / len(all_checks)) * 100 if all_checks else 0

print(f"\n🎯 Gateway Integration Status: {success_rate:.1f}% Complete")

if success_rate >= 80:
    print("\n🎉 Excellent! Your gateway integration is working well.")
    print("   ✅ Ready to proceed to Notebook 3: Multi-Agent System")
elif success_rate >= 60:
    print("\n⚠️  Good progress, but some issues need attention.")
    print("   🔧 Review failed validations above before proceeding")
else:
    print("\n❌ Multiple issues detected. Please review and fix:")
    print("   📋 Check AWS permissions, resource status, and configuration")

## Step 7: Save Gateway Configuration

Let's save the gateway configuration for use in subsequent notebooks.

In [ ]:
# Save gateway configuration for next notebooks
import json
from pathlib import Path

print("💾 Saving Gateway Configuration")
print("═" * 35)

# Prepare configuration data
gateway_session_config = {
    "timestamp": time.time(),
    "workshop_session": "notebook-02-gateway",
    "cognito": {
        "user_pool_id": cognito_config['user_pool_id'],
        "client_id": cognito_config['client_id'],
        "discovery_url": cognito_config['discovery_url'],
        "region": cognito_config['region']
        # Note: Not storing client_secret or access_token for security
    },
    "gateway": {
        "gateway_id": gateway_id,
        "gateway_url": gateway_url,
        "gateway_name": gateway_name
    },
    "iam": {
        "gateway_role_arn": gateway_role_arn,
        "role_name": role_name
    },
    "targets": created_targets,
    "backend_urls": backend_urls,
    "validation_results": validation_results,
    "success_rate": success_rate
}

# Save to configs directory
config_file = Path().absolute().parent / "configs" / "gateway_session.json"
config_file.parent.mkdir(exist_ok=True)

with open(config_file, 'w') as f:
    json.dump(gateway_session_config, f, indent=2, default=str)

print(f"✅ Configuration saved: {config_file}")

# Also update the main agent config with gateway URI
agent_config_path = Path().absolute().parent.parent.parent / "sre_agent" / "config" / "agent_config.yaml"

if agent_config_path.exists():
    try:
        import yaml
        
        with open(agent_config_path, 'r') as f:
            agent_config = yaml.safe_load(f)
        
        # Update gateway URI
        if 'gateway' not in agent_config:
            agent_config['gateway'] = {}
        
        agent_config['gateway']['uri'] = gateway_url
        
        with open(agent_config_path, 'w') as f:
            yaml.dump(agent_config, f, default_flow_style=False)
        
        print(f"✅ Updated agent config: {agent_config_path}")
        print(f"   Gateway URI: {gateway_url}")
        
    except Exception as e:
        print(f"⚠️  Could not update agent config: {e}")

# Save access token to .env file
env_file = Path().absolute().parent.parent.parent / "sre_agent" / ".env"
if access_token and env_file.exists():
    try:
        # Read existing .env content
        with open(env_file, 'r') as f:
            env_lines = f.readlines()
        
        # Update or add GATEWAY_ACCESS_TOKEN
        token_updated = False
        for i, line in enumerate(env_lines):
            if line.startswith('GATEWAY_ACCESS_TOKEN='):
                env_lines[i] = f'GATEWAY_ACCESS_TOKEN={access_token}\n'
                token_updated = True
                break
        
        if not token_updated:
            env_lines.append(f'\n# Gateway access token from Notebook 2\n')
            env_lines.append(f'GATEWAY_ACCESS_TOKEN={access_token}\n')
        
        # Write updated .env file
        with open(env_file, 'w') as f:
            f.writelines(env_lines)
        
        print(f"✅ Updated .env with gateway access token")
        
    except Exception as e:
        print(f"⚠️  Could not update .env file: {e}")

print(f"\n📋 Configuration Summary:")
print(f"   Gateway ID: {gateway_id}")
print(f"   Tools Available: {len(available_tools) if 'available_tools' in locals() else 'Unknown'}")
print(f"   Success Rate: {success_rate:.1f}%")
print(f"   Ready for Multi-Agent Integration: {'Yes' if success_rate >= 70 else 'Needs Review'}")

## Understanding What You've Built

Congratulations! You've successfully created a secure, production-ready gateway that transforms your backend APIs into standardized MCP tools.

In [ ]:
print("🎉 Gateway Integration Complete!")
print("═" * 40)

print("\n✅ What You've Accomplished:")
print("   • Created Amazon Cognito User Pool with OAuth 2.0")
print("   • Set up IAM roles with proper AgentCore permissions")
print("   • Deployed AgentCore Gateway with MCP protocol")
print("   • Transformed 4 backend APIs into standardized tools")
print("   • Implemented secure authentication and authorization")
print("   • Tested end-to-end tool invocation")

print("\n🏗️ Architecture Components Created:")
print("   • OAuth 2.0 Authentication: Secure agent access control")
print("   • AgentCore Gateway: MCP protocol implementation")
print("   • MCP Tools: Standardized interface to backend services")
print("   • Security Model: Inbound JWT + Outbound IAM authentication")

if 'available_tools' in locals() and available_tools:
    print(f"\n🔧 MCP Tools Created ({len(available_tools)} total):")
    tool_categories = {}
    for tool in available_tools:
        name = tool.get('name', '')
        if '___' in name:
            category = name.split('___')[0]
            tool_name = name.split('___')[1]
        else:
            category = 'Other'
            tool_name = name
        
        if category not in tool_categories:
            tool_categories[category] = []
        tool_categories[category].append(tool_name)
    
    for category, tools in tool_categories.items():
        print(f"   • {category}: {len(tools)} tools ({', '.join(tools[:2])}{'...' if len(tools) > 2 else ''})")

print(f"\n🚀 Ready for Next Steps:")
print(f"   📖 Notebook 3: Multi-Agent System")
print(f"      - Build LangGraph agent orchestration")
print(f"      - Connect agents to MCP tools")
print(f"      - Test realistic SRE investigations")

print(f"\n💡 Key Security Features:")
print(f"   • All communication over HTTPS with OAuth tokens")
print(f"   • Gateway validates all tool schemas and inputs")
print(f"   • IAM roles provide least-privilege access")
print(f"   • JWT tokens enable fine-grained access control")

print(f"\n🎯 MCP Protocol Benefits Achieved:")
print(f"   • Standardized tool interface across all backend APIs")
print(f"   • Automatic schema validation and error handling")
print(f"   • Agent-agnostic tool discovery and invocation")
print(f"   • Secure, scalable tool access management")

## Cleanup (Optional)

If you want to clean up AWS resources created in this notebook, run the cells below. **Note:** This will break the integration for subsequent notebooks.

In [ ]:
# Optional: Clean up AWS resources
# Uncomment and run if you want to clean up (this will break subsequent notebooks)

# print("🧹 Cleaning up AWS resources...")
# print("⚠️  WARNING: This will break integration for subsequent notebooks!")

# from workshop_utils import cleanup_cognito_resources, cleanup_iam_role

# # Delete gateway (this will also delete targets)
# try:
#     gateway_client.delete_gateway(gatewayIdentifier=gateway_id)
#     print(f"✅ Gateway {gateway_id} deleted")
# except Exception as e:
#     print(f"❌ Failed to delete gateway: {e}")

# # Clean up Cognito resources
# try:
#     cleanup_cognito_resources(cognito_config['user_pool_id'], region)
#     print(f"✅ Cognito resources cleaned up")
# except Exception as e:
#     print(f"❌ Failed to clean up Cognito: {e}")

# # Clean up IAM role
# try:
#     cleanup_iam_role(role_name)
#     print(f"✅ IAM role {role_name} deleted")
# except Exception as e:
#     print(f"❌ Failed to clean up IAM role: {e}")

print("💡 AWS resources are preserved for use in subsequent notebooks.")
print("   You can proceed directly to Notebook 3: Multi-Agent System")
print("\n🔗 Configuration has been saved for the next notebook to use.")

## Next Steps

🎯 **You've successfully created a secure gateway with MCP tools!**

**Continue to:** [Notebook 3: Multi-Agent System](03-multi-agent-system.ipynb)

In the next notebook, you'll:
- Build the complete LangGraph multi-agent system
- Connect your agents to the MCP tools you just created
- Run realistic SRE investigations with agent collaboration
- Test complex scenarios with multiple specialist agents

---

### 💭 Reflection Questions
- How does the MCP protocol simplify agent-tool integration?
- What are the security benefits of the gateway architecture?
- How would you extend this to support additional backend services?

### 🔗 Additional Resources
- [Model Context Protocol Specification](https://modelcontextprotocol.org/)
- [Amazon Bedrock AgentCore Gateway Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/agents.html)
- [OAuth 2.0 Client Credentials Flow](https://tools.ietf.org/html/rfc6749#section-4.4)